In [16]:
install.packages("RSQLite")
install.packages("DBI")

library(RSQLite)
library(DBI)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [17]:
library(readr)

customers <- read_csv("customers.csv")
orders <- read_csv("orders.csv")
deliveries <- read_csv("deliveries.csv")
drivers <- read_csv("drivers.csv")
vehicles <- read_csv("vehicles.csv")
hubs <- read_csv("hubs.csv")
complaints <- read_csv("complaints.csv")
incidents <- read_csv("incidents.csv")
app_events <- read_csv("app_events.csv")

Rows: 650 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (5): customer_id, home_zone, customer_type, preferred_channel, account_...
dbl  (3): age, loyalty_score, app_engagement_score
dttm (1): signup_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1250 Columns: 11
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): order_id, customer_id, service_type, pickup_zone, dropoff_zone, pr...
dbl  (3): promised_window_hours, order_value, special_handling_flag
dttm (1): order_created_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 950 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (6)

In [18]:
library(DBI)
library(RSQLite)

# Creating SQLite database connection
con <- dbConnect(SQLite(), "northstar_sql.db")

# Copying R data frames into SQLite tables
dbWriteTable(con, "customers", customers, overwrite = TRUE)
dbWriteTable(con, "orders", orders, overwrite = TRUE)
dbWriteTable(con, "deliveries", deliveries, overwrite = TRUE)
dbWriteTable(con, "drivers", drivers, overwrite = TRUE)
dbWriteTable(con, "vehicles", vehicles, overwrite = TRUE)
dbWriteTable(con, "hubs", hubs, overwrite = TRUE)
dbWriteTable(con, "complaints", complaints, overwrite = TRUE)
dbWriteTable(con, "incidents", incidents, overwrite = TRUE)
dbWriteTable(con, "app_events", app_events, overwrite = TRUE)

# Check tables created
dbListTables(con)

[1] "app_events" "complaints" "customers"  "deliveries" "drivers"   
[6] "hubs"       "incidents"  "orders"     "vehicles"

In [19]:
dbGetQuery(con, "
SELECT *
FROM deliveries
LIMIT 10
")

delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
DL00001,O00938,D004,V056,H05,1718708220,1718787960,Failed,17.26,1,0,3.07,12.05
DL00002,O00004,D138,V007,H02,1736621100,1736617140,OnTime,10.34,1,0,5.00,13.41
DL00003,O00639,D006,V049,H02,1748896740,1748900732,OnTime,7.92,0,0,4.98,8.51
DL00004,O00313,D116,V055,H02,1709940660,1710027008,Delayed,16.42,0,0,4.18,13.62
DL00005,O00844,D108,V034,H01,1758454980,1758469534,OnTime,14.52,1,0,4.18,9.22
DL00006,O00029,D037,V098,H03,1726058400,1726161112,Delayed,13.84,0,0,1.57,9.58
DL00007,O00097,D151,V037,H07,1704807660,1704929951,Delayed,32.72,0,0,4.64,17.70
DL00008,O00207,D082,V066,H03,1724362440,1724368942,OnTime,7.16,1,0,3.76,11.66
DL00009,O00297,D088,V029,H05,1712957580,1712971133,OnTime,40.23,1,0,3.70,15.78


In [20]:
dbGetQuery(con, "
SELECT delivery_id,
       delivery_status,
       route_distance_km,
       manual_route_override_count
FROM deliveries
WHERE delivery_status = 'Delayed'
LIMIT 10
")

delivery_id,delivery_status,route_distance_km,manual_route_override_count
<chr>,<chr>,<dbl>,<dbl>
DL00004,Delayed,16.42,0
DL00006,Delayed,13.84,0
DL00007,Delayed,32.72,0
DL00017,Delayed,20.79,0
DL00028,Delayed,15.53,2
DL00034,Delayed,14.68,0
DL00048,Delayed,7.37,0
DL00052,Delayed,26.38,1
DL00054,Delayed,29.80,1


In [21]:
dbGetQuery(con, "
PRAGMA table_info(deliveries)
")

cid,name,type,notnull,dflt_value,pk
<int>,<chr>,<chr>,<int>,<lgl>,<int>
0,delivery_id,TEXT,0,NA,0
1,order_id,TEXT,0,NA,0
2,driver_id,TEXT,0,NA,0
3,vehicle_id,TEXT,0,NA,0
4,hub_id,TEXT,0,NA,0
5,dispatch_time,REAL,0,NA,0
6,delivery_completed_at,REAL,0,NA,0
7,delivery_status,TEXT,0,NA,0
8,route_distance_km,REAL,0,NA,0


In [22]:
dbGetQuery(con, "
SELECT delivery_id,
       delivery_status,
       route_distance_km,
       manual_route_override_count
FROM deliveries
ORDER BY manual_route_override_count DESC
LIMIT 10
")

delivery_id,delivery_status,route_distance_km,manual_route_override_count
<chr>,<chr>,<dbl>,<dbl>
DL00473,OnTime,14.15,7
DL00055,Delayed,35.07,5
DL00085,OnTime,9.49,5
DL00374,OnTime,23.47,5
DL00672,OnTime,35.33,5
DL00744,Delayed,25.36,5
DL00881,Delayed,12.77,5
DL00922,Delayed,12.65,5
DL00019,OnTime,16.35,4


In [23]:
dbGetQuery(con, "
SELECT delivery_status,
       COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
")

delivery_status,total_deliveries
<chr>,<int>
Delayed,202
Failed,132
OnTime,616


In [24]:
dbGetQuery(con, "
SELECT delivery_status,
       AVG(route_distance_km) AS average_distance
FROM deliveries
GROUP BY delivery_status
")

delivery_status,average_distance
<chr>,<dbl>
Delayed,14.67025
Failed,13.36530
OnTime,13.77636


In [25]:
dbGetQuery(con, "
SELECT d.delivery_id,
       d.delivery_status,
       d.hub_id,
       h.hub_name
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
LIMIT 10
")

delivery_id,delivery_status,hub_id,hub_name
<chr>,<chr>,<chr>,<chr>
DL00001,Failed,H05,Central Core
DL00002,OnTime,H02,South Link
DL00003,OnTime,H02,South Link
DL00004,Delayed,H02,South Link
DL00005,OnTime,H01,North Exchange
DL00006,Delayed,H03,East Dock
DL00007,Delayed,H07,Riverside Hub
DL00008,OnTime,H03,East Dock
DL00009,OnTime,H05,Central Core


In [26]:
dbGetQuery(con, "
SELECT h.hub_name,
       d.delivery_status,
       COUNT(*) AS total_deliveries
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
GROUP BY h.hub_name, d.delivery_status
ORDER BY total_deliveries DESC
")

hub_name,delivery_status,total_deliveries
<chr>,<chr>,<int>
North Exchange,OnTime,93
East Dock,OnTime,85
West Gate,OnTime,83
Midtown Relay,OnTime,80
Riverside Hub,OnTime,76
South Link,OnTime,70
Central Core,OnTime,67
Airport Hub,OnTime,62
West Gate,Delayed,28


In [27]:
dbGetQuery(con, "
SELECT complaint_type,
       COUNT(*) AS total_complaints
FROM complaints
GROUP BY complaint_type
ORDER BY total_complaints DESC
")

complaint_type,total_complaints
<chr>,<int>
Delay,101
MissedPickup,64
AppIssue,53
DriverBehaviour,51
SupportExperience,20
Billing,16
Damage,15


In [28]:
dbGetQuery(con, "
SELECT h.hub_name,
       COUNT(*) AS total_delayed
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
WHERE d.delivery_status = 'Delayed'
GROUP BY h.hub_name
ORDER BY total_delayed DESC
LIMIT 5
")

hub_name,total_delayed
<chr>,<int>
West Gate,28
Airport Hub,27
South Link,26
North Exchange,26
Riverside Hub,25
